# 🧪 Emoji Lyrics Overlay Test

Standalone diagnostic notebook for the Lyrics Video Tool.

**Purpose:** test emoji asset discovery, mapping, transparency and rendering without audio, Whisper, Stage 1 timing, or the Stage 2 notebook.

Existing `AIgenerated/assets/emoji/` assets are READ ONLY. Test outputs go into `AIgenerated/emoji_test/`. 

In [ ]:
from google.colab import drive
import os, re, json, shutil, traceback
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

drive.mount('/content/drive', force_remount=False)

FOLDER='/content/drive/MyDrive/AIgenerated'
EMOJI_ROOT=os.path.join(FOLDER,'assets','emoji')
TEST_ROOT=os.path.join(FOLDER,'emoji_test')
INDIV_ROOT=os.path.join(TEST_ROOT,'individual')
SHEET_PATH=os.path.join(TEST_ROOT,'emoji_test_sheet.png')
REPORT_PATH=os.path.join(TEST_ROOT,'emoji_test_report.json')

os.makedirs(TEST_ROOT,exist_ok=True)
os.makedirs(INDIV_ROOT,exist_ok=True)

print('Workspace:',FOLDER)
print('Emoji assets:',EMOJI_ROOT)
print('Test output:',TEST_ROOT)


In [ ]:
# Find a usable image in AIgenerated/ as the visual canvas.
IMAGE_EXTS={'.png','.jpg','.jpeg','.webp'}

def natural_key(p):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)',Path(p).name)]

candidates=[]
for p in sorted(Path(FOLDER).iterdir(),key=natural_key):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS and not p.name.startswith('_'):
        candidates.append(str(p))

if not candidates:
    raise RuntimeError('No usable image found directly inside AIgenerated/.')

BASE_IMAGE=candidates[0]
print('Canvas image:',BASE_IMAGE)
with Image.open(BASE_IMAGE) as im:
    print('Canvas size:',im.size,'mode:',im.mode)


In [ ]:
# Discover every emoji asset recursively.
ASSET_EXTS={'.png','.webp','.jpg','.jpeg'}

if not os.path.isdir(EMOJI_ROOT):
    raise FileNotFoundError(f'Emoji asset folder not found: {EMOJI_ROOT}')

ASSETS=[]
for p in sorted(Path(EMOJI_ROOT).rglob('*'),key=natural_key):
    if p.is_file() and p.suffix.lower() in ASSET_EXTS:
        try:
            with Image.open(p) as im:
                im.load()
                rgba=im.convert('RGBA')
                alpha=rgba.getchannel('A')
                bbox=alpha.getbbox()
                transparent=(bbox is not None and bbox != (0,0,rgba.width,rgba.height))
                ASSETS.append({
                    'path':str(p),
                    'name':p.name,
                    'relative':str(p.relative_to(EMOJI_ROOT)),
                    'size':[im.width,im.height],
                    'mode':im.mode,
                    'has_alpha':('A' in im.getbands()),
                    'has_transparency':transparent
                })
        except Exception as e:
            ASSETS.append({'path':str(p),'name':p.name,'relative':str(p.relative_to(EMOJI_ROOT)),'error':str(e)})

print('Emoji assets found:',len(ASSETS))
for i,a in enumerate(ASSETS,1):
    print(f"{i:03d}. {a['relative']} | {a.get('size')} | alpha={a.get('has_alpha')}")


In [ ]:
# Try to infer the emoji represented by common filename conventions.
# This is diagnostic only; no assumptions are used by Stage 2.

def infer_label(name):
    stem=Path(name).stem
    # Unicode forms: U+1F525, 1F525, unicode_1F525, etc.
    m=re.search(r'(?:U\+|unicode[_-]?)?([0-9A-Fa-f]{4,6})(?:[-_][0-9A-Fa-f]{4,6})*',stem)
    codepoints=[]
    if m:
        for token in re.findall(r'[0-9A-Fa-f]{4,6}',m.group(0)):
            try: codepoints.append(chr(int(token,16)))
            except Exception: pass
    return ''.join(codepoints) if codepoints else ''

for a in ASSETS:
    a['inferred_emoji']=infer_label(a['name'])

print('Filename inference preview:')
for a in ASSETS[:100]:
    print(a['name'],'=>',repr(a['inferred_emoji']))


In [ ]:
# Render each asset independently on the same canvas.
# We intentionally use the asset itself; there is no font-based emoji rendering.

with Image.open(BASE_IMAGE) as src:
    canvas_base=src.convert('RGBA')

MAX_EMOJI_SIZE=360
results=[]

for idx,a in enumerate(ASSETS,1):
    out_name=f'{idx:03d}_{Path(a["name"]).stem}_test.png'
    out_path=os.path.join(INDIV_ROOT,out_name)
    row={'index':idx,'asset':a.copy(),'output':out_path,'status':'failed'}
    try:
        if 'error' in a:
            raise RuntimeError(a['error'])
        with Image.open(a['path']) as e:
            emoji=e.convert('RGBA')
        emoji.thumbnail((MAX_EMOJI_SIZE,MAX_EMOJI_SIZE),Image.Resampling.LANCZOS)
        # Place near center with a semi-transparent diagnostic panel.
        canvas=canvas_base.copy()
        panel=Image.new('RGBA',(500,500),(0,0,0,145))
        canvas.alpha_composite(panel,((canvas.width-500)//2,(canvas.height-500)//2))
        x=(canvas.width-emoji.width)//2
        y=(canvas.height-emoji.height)//2
        canvas.alpha_composite(emoji,(x,y))
        draw=ImageDraw.Draw(canvas)
        label=f"#{idx}  {a['name']}"
        draw.text((20,20),label,fill=(255,255,255,255))
        canvas.convert('RGB').save(out_path,quality=95)
        row['status']='rendered'
        row['render_size']=[emoji.width,emoji.height]
    except Exception as e:
        row['error']=traceback.format_exc()
    results.append(row)

print('Rendered:',sum(r['status']=='rendered' for r in results))
print('Failed:',sum(r['status']=='failed' for r in results))


In [ ]:
# Build a contact sheet so dozens of emoji can be visually verified at once.

THUMB_W,THUMB_H=270,360
COLS=4
ROWS=max(1,(len(results)+COLS-1)//COLS)
sheet=Image.new('RGB',(COLS*THUMB_W,ROWS*THUMB_H),(35,35,35))
draw=ImageDraw.Draw(sheet)

for i,r in enumerate(results):
    cx=(i%COLS)*THUMB_W
    cy=(i//COLS)*THUMB_H
    if r['status']=='rendered' and os.path.isfile(r['output']):
        with Image.open(r['output']) as im:
            thumb=im.convert('RGB')
            thumb.thumbnail((THUMB_W-10,THUMB_H-55),Image.Resampling.LANCZOS)
            sheet.paste(thumb,(cx+(THUMB_W-thumb.width)//2,cy+5))
        status='OK'
    else:
        status='FAILED'
    draw.text((cx+8,cy+THUMB_H-45),f"#{r['index']} {r['asset']['name'][:28]}",fill='white')
    draw.text((cx+8,cy+THUMB_H-25),status,fill='white')

sheet.save(SHEET_PATH)
print('Contact sheet:',SHEET_PATH)


In [ ]:
# Save a machine-readable diagnostic report.

report={
    'base_image':BASE_IMAGE,
    'emoji_root':EMOJI_ROOT,
    'asset_count':len(ASSETS),
    'rendered_count':sum(r['status']=='rendered' for r in results),
    'failed_count':sum(r['status']=='failed' for r in results),
    'results':results,
}

with open(REPORT_PATH,'w',encoding='utf-8') as f:
    json.dump(report,f,ensure_ascii=False,indent=2)

print('Report:',REPORT_PATH)
print('DONE')
print('Open the contact sheet and inspect whether the emoji are visibly overlaid.')
